# `sparsegf2.circuits.measurements`, line by line

This notebook **rewrites and explains** every line of
[`src/sparsegf2/circuits/measurements.py`](../../src/sparsegf2/circuits/measurements.py).

## Where this module sits

A graph-defined MIPT circuit alternates two things each layer:

1. a **gate layer**, two-qubit Cliffords on the edges chosen by the
   scheduler, and
2. a **measurement layer**, single-qubit $Z$ measurements on a random
   subset of the $n$ system qubits.

This module owns **step 2's qubit selection**: given the mode, $n$, the
probability $p$, and the gate pairs that just fired, it returns the sorted
list of qubit indices to measure. It does **not** perform the measurement
or decide its outcome; that happens in
`SparseGF2.measure_z`, driven by a *separate* RNG stream (see the runner
notebook). Keeping *selection* and *outcome* apart is a deliberate design
choice: you can replay the same circuit with different measurement coins
by changing only the outcome stream.

The three modes:

| mode | candidate set each layer | RNG draws |
|---|---|---|
| `bernoulli` | all $n$ qubits, each kept w.p. $p$ | one length-$n$ uniform vector |
| `gated` | only qubits the gate layer touched, each kept w.p. $p$ | one length-(#touched) vector |
| `random_pair` | exactly 2 random qubits, each kept w.p. $p$ | a 2-choice + 2 coins |

## Setup

Import the module and a seeded RNG so every demo below is reproducible.

In [1]:
import numpy as np
from sparsegf2.circuits.measurements import MEASUREMENT_MODES, sample_measurements
from sparsegf2.errors import InvalidArgumentError

MEASUREMENT_MODES

('bernoulli', 'gated', 'random_pair', 'uniform_count')

## The mode registry

```python
MEASUREMENT_MODES: tuple[str, ...] = ("bernoulli", "gated", "random_pair")
```

A module-level tuple is the single source of truth for the valid mode
names. `CircuitConfig` validates against it, and `sample_measurements`
rejects anything outside it. The mode is named **`bernoulli`** rather than
`"uniform"` because "uniform" is overloaded (it also suggests *uniform-count*),
and "bernoulli" names exactly what it is: an independent Bernoulli($p$)
trial per qubit.

## The function signature

```python
def sample_measurements(mode, n, p, gate_pairs, rng) -> list[int]:
```

- `mode`: one of `MEASUREMENT_MODES`.
- `n`: number of **system** qubits (the candidate universe is `0..n-1`).
- `p`: per-qubit keep probability, in $[0, 1]$.
- `gate_pairs`: the pairs that fired this layer; only `gated` reads them.
- `rng`: a `numpy.random.Generator` passed in by the scheduler, so the
  draw order is controlled centrally (load-bearing for reproducibility).

It returns a **sorted, deduplicated** `list[int]`.

## Input validation

```python
if mode not in MEASUREMENT_MODES:
    raise InvalidArgumentError(...)
if not 0.0 <= p <= 1.0:
    raise InvalidArgumentError(...)
```

Both raise `InvalidArgumentError`, which multi-inherits `ValueError`, so
existing `except ValueError` handlers still catch it, but the package's
own exception type lets callers distinguish *our* bad-input errors from a
stray numpy `ValueError`. Let's see both fire:

In [2]:
for bad in [('nope', 8, 0.1), ('bernoulli', 8, 1.5)]:
    try:
        sample_measurements(bad[0], bad[1], bad[2], [], np.random.default_rng(0))
    except InvalidArgumentError as e:
        print('rejected:', e)
        assert isinstance(e, ValueError)  # backward-compatible


rejected: measurement_mode must be one of ('bernoulli', 'gated', 'random_pair', 'uniform_count'); got 'nope'
rejected: p must be in [0, 1]; got 1.5


## Branch 1: `bernoulli`

```python
if mode == "bernoulli":
    draws = rng.random(n)
    return np.nonzero(draws < p)[0].tolist()
```

Line by line:

- `rng.random(n)` draws `n` i.i.d. uniforms in $[0, 1)$ in **one
  vectorized call**, far cheaper than a Python loop of `n` scalar draws.
- `draws < p` is a boolean mask: `True` exactly where that qubit's coin
  landed below $p$ (probability $p$).
- `np.nonzero(...)[0]` returns the indices of the `True` entries, **already
  in ascending order**, so no explicit `sorted()` is needed.
- `.tolist()` converts the numpy index array to a Python `list[int]` in C.

That last step is an optimization a benchmark confirmed: `.tolist()`
beats the previous `sorted(int(q) for q in ...)` comprehension by 3-6×
on this op. Let's reproduce that head-to-head:

In [3]:
import timeit
n = 256
rng = np.random.default_rng(0)
draws = rng.random(n)
p = 0.1
old = lambda: sorted(int(q) for q in np.nonzero(draws < p)[0])
new = lambda: np.nonzero(draws < p)[0].tolist()
assert old() == new()  # identical output
t_old = timeit.timeit(old, number=20000)
t_new = timeit.timeit(new, number=20000)
print(f'old (sorted+int comprehension): {t_old*1e3:.1f} ms / 20k')
print(f'new (np .tolist):               {t_new*1e3:.1f} ms / 20k')
print(f'speedup: {t_old/t_new:.1f}x')


old (sorted+int comprehension): 74.6 ms / 20k
new (np .tolist):               22.8 ms / 20k
speedup: 3.3x


**Sanity demos** for `bernoulli`, the boundary cases $p=0$ and $p=1$:

In [4]:
rng = np.random.default_rng(1)
print('p=0 :', sample_measurements('bernoulli', 8, 0.0, [], rng))   # never measure
print('p=1 :', sample_measurements('bernoulli', 8, 1.0, [], rng))   # always all qubits
out = sample_measurements('bernoulli', 16, 0.5, [], rng)
print('p=.5:', out)
assert out == sorted(set(out))  # sorted + unique, by construction


p=0 : []
p=1 : [0, 1, 2, 3, 4, 5, 6, 7]
p=.5: [0, 1, 2, 3, 5, 6, 11, 12, 15]


## Branch 2: `gated`

```python
if mode == "gated":
    candidates = sorted({int(q) for pair in gate_pairs for q in pair})
    if not candidates:
        return []
    draws = rng.random(len(candidates))
    return sorted(candidates[i] for i in range(len(candidates)) if draws[i] < p)
```

Here the candidate universe is **only the qubits this layer's gates
touched**, not all $n$:

- the set comprehension flattens `gate_pairs` into the distinct qubits
  involved (a `set` dedups, e.g. if two gates shared a qubit, which
  brickwork never does, but `random_edge`-style overlaps could);
- `sorted(...)` fixes a deterministic candidate order so the RNG draws map
  to qubits reproducibly;
- if no gates fired (`candidates` empty), return `[]` immediately, with no RNG
  consumed;
- otherwise draw one uniform per candidate and keep those below $p$.

With $p=1$ every touched qubit is measured:

In [5]:
rng = np.random.default_rng(2)
pairs = [(0, 1), (4, 5)]
print('gated p=1, pairs', pairs, '->', sample_measurements('gated', 8, 1.0, pairs, rng))
print('gated, no gates  ->', sample_measurements('gated', 8, 1.0, [], rng))


gated p=1, pairs [(0, 1), (4, 5)] -> [0, 1, 4, 5]
gated, no gates  -> []


## Branch 3: `random_pair`

```python
if n < 2:
    return []
pair = rng.choice(n, size=2, replace=False)
draws = rng.random(2)
kept = [int(pair[i]) for i in range(2) if draws[i] < p]
return sorted(set(kept))
```

The candidate set is **exactly two distinct qubits**, chosen uniformly
without replacement, each then passed through the same Bernoulli($p$)
gate:

- `n < 2` guards the degenerate case (can't pick 2 distinct qubits);
- `rng.choice(n, size=2, replace=False)` is the uniform 2-subset draw;
- the two coins decide which of the pair survive;
- `sorted(set(...))` keeps the output sorted + unique (the `set` is belt
  and braces, since the pair is already distinct).

At $p=1$ both are always kept, so the result has length 2:

In [6]:
rng = np.random.default_rng(3)
for _ in range(4):
    print(sample_measurements('random_pair', 8, 1.0, [], rng))
print('n<2 ->', sample_measurements('random_pair', 1, 1.0, [], rng))


[0, 5]
[0, 1]
[1, 7]
[5, 7]
n<2 -> []


## Determinism: the contract the scheduler relies on

`sample_measurements` draws from the `rng` you hand it, in a fixed order.
So the **same generator state** produces the **same selection**, which is
what makes a whole sweep reproducible from `(base_seed, sample_seed,
layer_index)`. Two generators seeded identically agree exactly:

In [7]:
a = np.random.default_rng(12345)
b = np.random.default_rng(12345)
seq_a = [sample_measurements('bernoulli', 32, 0.3, [], a) for _ in range(5)]
seq_b = [sample_measurements('bernoulli', 32, 0.3, [], b) for _ in range(5)]
assert seq_a == seq_b
print('identical across two equally-seeded RNGs:', seq_a == seq_b)
print('first layer selection:', seq_a[0])


identical across two equally-seeded RNGs: True
first layer selection: [0, 7, 10, 13, 19, 20, 21, 24, 26, 27, 28]


## Mean check: does `bernoulli` actually keep $\approx np$ qubits?

A quick empirical confirmation that the Bernoulli($p$) semantics hold: the
average number of measured qubits per layer should sit near $n p$.

In [8]:
rng = np.random.default_rng(7)
n, p, trials = 64, 0.2, 2000
counts = [len(sample_measurements('bernoulli', n, p, [], rng)) for _ in range(trials)]
print(f'empirical mean per layer: {np.mean(counts):.3f}   expected n*p = {n*p}')
assert abs(np.mean(counts) - n * p) < 0.5


empirical mean per layer: 12.714   expected n*p = 12.8


## Summary

- `measurements.py` is **pure selection logic**: no simulator coupling,
  no measurement outcomes.
- Three modes, one validated entry point, deterministic given the RNG.
- The only optimization (`bernoulli` → `np.nonzero(...).tolist()`) is
  benchmark-justified and semantically identical to the obvious version.

Next module: the **scheduler**, which calls this function once per layer
with the RNG it owns (see `notebooks/circuits/scheduler.ipynb`).